# AWS Financial AI Agent - Invocation Demo

This notebook demonstrates end-to-end user authentication, live agent invocation, and observability trace extraction **without requiring any AWS access keys** from the recruiter. Credentials and secrets are retrieved securely via Cognito Identity Pools and AWS SSM.

In [ ]:
!pip install boto3 requests

import boto3
import requests
import json
import urllib.parse

REGION = "us-east-1"
USER_POOL_ID = "us-east-1_5pCxpIkx8"
CLIENT_ID = "2r1ik1k110jbu6nfmuoegk5lns"
IDENTITY_POOL_ID = "us-east-1:c7680c24-fe96-4358-b305-6f43de1ca6c8"

access_token = None
id_token = None
username = None
password = None

### 1. Retrieve Guest Login Credentials
We use the Cognito Identity Pool (Unauthenticated) to securely fetch the login credentials from AWS SSM.

In [ ]:
identity_client = boto3.client('cognito-identity', region_name=REGION)

try:
    # 1. Get Guest Identity ID
    id_resp = identity_client.get_id(IdentityPoolId=IDENTITY_POOL_ID)
    identity_id = id_resp['IdentityId']
    
    # 2. Get Guest Credentials
    cred_resp = identity_client.get_credentials_for_identity(IdentityId=identity_id)
    guest_creds = cred_resp['Credentials']
    
    # 3. Fetch Username/Password from SSM
    ssm_guest = boto3.client(
        'ssm', region_name=REGION,
        aws_access_key_id=guest_creds['AccessKeyId'],
        aws_secret_access_key=guest_creds['SecretKey'],
        aws_session_token=guest_creds['SessionToken']
    )
    
    username = ssm_guest.get_parameter(Name='/financial-ai/auth/analyst-username', WithDecryption=True)['Parameter']['Value']
    password = ssm_guest.get_parameter(Name='/financial-ai/auth/analyst-password', WithDecryption=True)['Parameter']['Value']
    
    print(f"✅ Successfully retrieved demo credentials via Guest Identity.")
except Exception as e:
    print(f"❌ Guest Retrieval Failed: {str(e)}")

### 2. Authenticate with Amazon Cognito
Login using the credentials retrieved in the previous step.

In [ ]:
client = boto3.client('cognito-idp', region_name=REGION)
try:
    auth_response = client.initiate_auth(
        ClientId=CLIENT_ID,
        AuthFlow='USER_PASSWORD_AUTH',
        AuthParameters={'USERNAME': username, 'PASSWORD': password}
    )
    access_token = auth_response['AuthenticationResult']['AccessToken']
    id_token = auth_response['AuthenticationResult']['IdToken']
    print(f"✅ Cognito Authentication Successful.")
except Exception as e:
    print(f"❌ Authentication Failed: {str(e)}")

### 3. Invoke the Agentcore Streaming Endpoint

In [ ]:
AGENT_ARN = "arn:aws:bedrock-agentcore:us-east-1:162187491349:runtime/Financial_Analyst_Agent-s5aas5HZhy"
encoded_arn = urllib.parse.quote(AGENT_ARN, safe='')
AGENTCORE_URL = f"https://bedrock-agentcore.us-east-1.amazonaws.com/runtimes/{encoded_arn}/invocations"
SESSION_ID = "demo-session-recruiters-verification-proof-2026"

def query_financial_agent(prompt: str):
    if access_token is None:
        print("❌ ERROR: Access token is missing. Run previous cells successfully first.")
        return

    print(f"\n--- Query: {prompt} ---")
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
        "Accept": "text/event-stream",
        "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": SESSION_ID
    }

    response = requests.post(AGENTCORE_URL, headers=headers, json={"prompt": prompt}, stream=True)
    if response.status_code != 200:
        print(f"❌ Error {response.status_code}: {response.text}")
        return

    for line in response.iter_lines():
        if line:
            decoded = line.decode('utf-8')
            if decoded.startswith("data: "):
                data = json.loads(decoded[6:])
                print(data.get("event", ""), end="", flush=True)

### 4. Execute Required Financial Queries

In [ ]:
queries = [
    "What is the stock price for Amazon right now?",
    "What were the stock prices for Amazon in Q4 last year?",
    "Compare Amazon's recent stock performance to what analysts predicted in their reports.",
    "I’m researching AMZN give me the current price and any relevant information about their AI business.",
    "What is the total amount of office space Amazon owned in North America in 2024?"
]
for q in queries: query_financial_agent(q)

### 5. Observability Verification (Authenticated AWS Identity)
Fetch Langfuse traces securely using temporary credentials obtained via the Cognito session.

In [ ]:
try:
    # Exchange IdToken for Authenticated credentials
    id_resp = identity_client.get_id(
        IdentityPoolId=IDENTITY_POOL_ID,
        Logins={ f'cognito-idp.{REGION}.amazonaws.com/{USER_POOL_ID}': id_token }
    )
    auth_creds = identity_client.get_credentials_for_identity(
        IdentityId=id_resp['IdentityId'],
        Logins={ f'cognito-idp.{REGION}.amazonaws.com/{USER_POOL_ID}': id_token }
    )['Credentials']
    
    ssm_auth = boto3.client(
        'ssm', region_name=REGION,
        aws_access_key_id=auth_creds['AccessKeyId'],
        aws_secret_access_key=auth_creds['SecretKey'],
        aws_session_token=auth_creds['SessionToken']
    )
    
    pk = ssm_auth.get_parameter(Name='/financial-ai/langfuse/public-key', WithDecryption=True)['Parameter']['Value']
    sk = ssm_auth.get_parameter(Name='/financial-ai/langfuse/secret-key', WithDecryption=True)['Parameter']['Value']
    
    print(f"✅ Successfully retrieved Langfuse keys.")
    trace_resp = requests.get(f"https://cloud.langfuse.com/api/public/traces?sessionId={SESSION_ID}", auth=(pk, sk))
    if trace_resp.status_code == 200:
        print(f"\n--- Langfuse Trace Audit ---")
        print(json.dumps(trace_resp.json(), indent=2))
except Exception as e:
    print(f"❌ Observability Retrieval Failed: {str(e)}")